In [ ]:
#config

ESOL_CSV = '/kaggle/input/datasets/kaalmurlidhar/esolandfreesol/delaney-processed.csv'

# training config — copied verbatim from finetune_regression_ESOL.py
ESOL_EPOCHS         = 100
ESOL_LR             = 2e-5
ESOL_BS             = 16
ESOL_WEIGHT_DECAY   = 0.0   # original does not set weight_decay
ESOL_SEED           = 2024
ESOL_EARLY_STOP     = 50
ESOL_LR_FACTOR      = 0.6
ESOL_LR_PATIENCE    = 10
ESOL_LR_MIN         = 5e-6

# shared model architecture — from finetune_regression_ESOL.py
D_MODEL    = 768
N_LAYERS   = 6
N_HEADS    = 12
D_K        = 64
D_V        = 64
D_FF       = 768 * 4
VOCAB_SIZE = 83
MAXLEN     = 501
GCN_EMB_DIM  = 300
GCN_FEAT_DIM = 300
COMBINED_DIM = D_MODEL + GCN_FEAT_DIM  # 1068

# electron token sets


# correct: heteroatoms (N, O, S), aromatic atoms, pi bonds.
# verified against preprocess/property_tags.py.
ELECTRON_RELEVANT_TOKEN_SET = {7, 8, 9, 15, 16, 17, 18, 57, 58, 78}

# wrong: structurally common tokens with NO electron chemistry significance.
# token 1:  smiles -> chain (backbone rule)
# token 3:  atom -> aliphatic_organic (structural rule)
# token 5:  aliphatic_organic -> 'B' (boron — rare, not electron-sensitive)
# token 6:  aliphatic_organic -> 'C' (carbon — the "boring" workhorse)
# token 38: DIGIT -> '1' (ring closure number)
# token 39: DIGIT -> '2'
# token 40: DIGIT -> '3'
# token 41: DIGIT -> '4'
# token 74: chain -> branched_atom (structural building block)
# token 75: chain -> chain branched_atom (structural extension)
# this set has the same cardinality (10 tokens) as electron_relevant_token_set
# to control for the effect of set size.
WRONG_CHEMISTRY_TOKEN_SET = {1, 3, 5, 6, 38, 39, 40, 41, 74, 75}

# ablation variant definitions
# each dict defines one variant. the run_esol_variant() function
# reads these parameters to configure the Embedding class and gcn.

ABLATION_VARIANTS = [
    {
        'id':              0,
        'name':            'baseline (learnable bias, correct chemistry)',
        'short':           'baseline',
        'bias_init':       0.1,
        'bias_learnable':  True,
        'token_set':       ELECTRON_RELEVANT_TOKEN_SET,
        'gcn_layers':      5,
        'gcn_drop':        0.1,
        'predictor_drop':  0.1,
    },
    {
        'id':              1,
        'name':            'ablation a: no bias — null hypothesis',
        'short':           'no_bias',
        'bias_init':       0.0,
        'bias_learnable':  False,   # frozen at 0.0
        'token_set':       ELECTRON_RELEVANT_TOKEN_SET,
        'gcn_layers':      5,
        'gcn_drop':        0.1,
        'predictor_drop':  0.1,
    },
    {
        'id':              2,
        'name':            'ablation b: wrong chemistry — control group',
        'short':           'wrong_chem',
        'bias_init':       0.1,
        'bias_learnable':  True,
        'token_set':       WRONG_CHEMISTRY_TOKEN_SET,   # C, boron, chain rules
        'gcn_layers':      5,
        'gcn_drop':        0.1,
        'predictor_drop':  0.1,
    },
    {
        'id':              3,
        'name':            'ablation c: static bias — learnable=False',
        'short':           'static_bias',
        'bias_init':       0.1,
        'bias_learnable':  False,   # frozen at 0.1
        'token_set':       ELECTRON_RELEVANT_TOKEN_SET,
        'gcn_layers':      5,
        'gcn_drop':        0.1,
        'predictor_drop':  0.1,
    },
    {
        'id':              4,
        'name':            'zone 3: architecture tuning (gcn=3, drop=0.2)',
        'short':           'arch_tuned',
        'bias_init':       0.1,
        'bias_learnable':  True,
        'token_set':       ELECTRON_RELEVANT_TOKEN_SET,
        'gcn_layers':      3,       # was 5. reduces over-smoothing.
        'gcn_drop':        0.1,
        'predictor_drop':  0.2,     # was 0.1. more regularization.
    },
]

# dependency installation

import subprocess
import sys
import torch as _torch_check

def _install_pyg_wheels():
    torch_version = _torch_check.__version__.split('+')[0]
    cuda_str = ('cu' + _torch_check.version.cuda.replace('.', '')) if _torch_check.cuda.is_available() else 'cpu'
    wheel_url = f'https://data.pyg.org/whl/torch-{torch_version}+{cuda_str}.html'
    print(f'  using pyg wheel index: {wheel_url}')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                           'torch_scatter', 'torch_sparse', '-f', wheel_url])

def _install_if_missing(import_name, pip_name, fallback_pip_name=None):
    try:
        __import__(import_name)
        print(f'  {import_name}: already installed, skipping.')
    except ImportError:
        print(f'  installing {pip_name}...')
        try:
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pip_name])
        except subprocess.CalledProcessError:
            if fallback_pip_name:
                subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', fallback_pip_name])
            else:
                raise

print('installing dependencies...')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'torch_geometric'])
_install_pyg_wheels()
_install_if_missing('rdkit', 'rdkit', fallback_pip_name='rdkit-pypi')
_install_if_missing('nltk',  'nltk')
_install_if_missing('six',   'six')

import nltk
nltk.download('punkt', quiet=True)
print('all dependencies installed.')

#  imports

import os
import sys
import copy
import math
import time
import contextlib
import re as _re
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split, Subset

from rdkit import Chem
from rdkit.Chem.rdchem import BondType as BT
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')

import six

from torch_geometric.nn import global_mean_pool
from torch_geometric.utils import add_self_loops
from torch_scatter import scatter_add

# zinc cfg grammar and encoding

GRAM_STRING = """smiles -> chain
atom -> bracket_atom
atom -> aliphatic_organic
atom -> aromatic_organic
aliphatic_organic -> 'B'
aliphatic_organic -> 'C'
aliphatic_organic -> 'N'
aliphatic_organic -> 'O'
aliphatic_organic -> 'S'
aliphatic_organic -> 'P'
aliphatic_organic -> 'F'
aliphatic_organic -> 'I'
aliphatic_organic -> 'Cl'
aliphatic_organic -> 'Br'
aromatic_organic -> 'c'
aromatic_organic -> 'n'
aromatic_organic -> 'o'
aromatic_organic -> 's'
bracket_atom -> '[' BAI ']'
BAI -> isotope symbol BAC
BAI -> symbol BAC
BAI -> isotope symbol
BAI -> symbol
BAC -> chiral BAH
BAC -> BAH
BAC -> chiral
BAH -> hcount BACH
BAH -> BACH
BAH -> hcount
BACH -> charge class
BACH -> charge
BACH -> class
symbol -> aliphatic_organic
symbol -> aromatic_organic
isotope -> DIGIT
isotope -> DIGIT DIGIT
isotope -> DIGIT DIGIT DIGIT
DIGIT -> '1'
DIGIT -> '2'
DIGIT -> '3'
DIGIT -> '4'
DIGIT -> '5'
DIGIT -> '6'
DIGIT -> '7'
DIGIT -> '8'
chiral -> '@'
chiral -> '@@'
hcount -> 'H'
hcount -> 'H' DIGIT
charge -> '-'
charge -> '-' DIGIT
charge -> '-' DIGIT DIGIT
charge -> '+'
charge -> '+' DIGIT
charge -> '+' DIGIT DIGIT
bond -> '-'
bond -> '='
bond -> '#'
bond -> '/'
bond -> '\\\\'
bond -> '.'
ringbond -> DIGIT
ringbond -> bond DIGIT
branched_atom -> atom
branched_atom -> atom RB
branched_atom -> atom BB
branched_atom -> atom RB BB
RB -> RB ringbond
RB -> ringbond
BB -> BB branch
BB -> branch
branch -> '(' chain ')'
branch -> '(' bond chain ')'
chain -> branched_atom
chain -> chain branched_atom
chain -> chain bond branched_atom
symbol -> element_symbols
aromatic_organic -> 'p'
element_symbols -> 'H'
class -> DIGIT
Nothing -> None"""

GCFG = nltk.CFG.fromstring(GRAM_STRING)
_RULE_LOOKUP = {}
for _idx, _prod in enumerate(GCFG.productions()):
    _lhs = _prod.lhs().symbol()
    _rhs = tuple(s.symbol() if not isinstance(s, str) else s for s in _prod.rhs())
    _RULE_LOOKUP[(_lhs, _rhs)] = _idx + 1

_PARSER = nltk.ChartParser(GCFG)
_SMILES_RE = _re.compile(r'(\[[^\]]+\]|Br|Cl|@@|[BCNOPSFIbcnosp=#@+\-\/\\\.1-9])')
_GRAMMAR_TOKEN_SET = set(range(1, 81))


def _tokenize(smiles):
    return _SMILES_RE.findall(smiles)


def encode_smiles(smiles):
    tokens = _tokenize(smiles)
    if not tokens:
        raise ValueError(f'tokenizer returned empty: {smiles}')
    trees = list(_PARSER.parse(tokens))
    if not trees:
        raise ValueError(f'cfg parse failed: {smiles}')
    rule_indices = []
    for subtree in trees[0].subtrees():
        if subtree.height() > 1:
            lhs = subtree.label()
            rhs = tuple(child.label() if isinstance(child, nltk.Tree) else child for child in subtree)
            idx = _RULE_LOOKUP.get((lhs, rhs))
            if idx is not None:
                rule_indices.append(idx)
    return rule_indices


def construct_token_sequence(rule_index_list, max_len=500):
    if len(rule_index_list) > max_len:
        rule_index_list = rule_index_list[:max_len]
    tokens = ['[GLO]'] + rule_index_list + ['[PAD]'] * (max_len - len(rule_index_list))
    tokens_idx, atom_mask = [], []
    for token in tokens:
        if token in _GRAMMAR_TOKEN_SET:
            atom_mask.append(1); tokens_idx.append(token)
        elif token == '[GLO]':
            atom_mask.append(1); tokens_idx.append(82)
        else:
            atom_mask.append(0); tokens_idx.append(0)
    return tokens_idx, atom_mask


# model architecture

# the Embedding class is parameterized to support all five ablation variants.
# bias_init and bias_learnable control the electron_bias parameter.
# electron_token_set selects which tokens receive the bias.

def get_attn_pad_mask(seq_q):
    batch_size, seq_len = seq_q.size()
    return seq_q.data.eq(0).unsqueeze(1).expand(batch_size, seq_len, seq_len)


class Embedding(nn.Module):
    # parameterized property-aware embedding.
    #
    # bias_init     : initial scalar value for electron_bias.
    # bias_learnable: if False, requires_grad=False. the optimizer cannot
    #                 update this parameter. used for ablations a (init=0)
    #                 and c (init=0.1, frozen).
    # electron_token_set: which token integers receive the bias. used for
    #                 ablation b to swap in the chemically inert control set.

    def __init__(self, vocab_size, d_model, maxlen,
                 bias_init=0.1, bias_learnable=True, electron_token_set=None):
        super().__init__()
        if electron_token_set is None:
            electron_token_set = ELECTRON_RELEVANT_TOKEN_SET

        self.tok_embed = nn.Embedding(vocab_size, d_model)
        self.pos_embed = nn.Embedding(maxlen, d_model)
        self.norm      = nn.LayerNorm(d_model)

        # the electron_bias parameter. configured per ablation variant.
        # requires_grad=False means the optimizer never touches it.
        self.electron_bias = nn.Parameter(
            torch.tensor(float(bias_init)),
            requires_grad=bias_learnable
        )

        self.register_buffer(
            'electron_token_indices',
            torch.tensor(sorted(electron_token_set), dtype=torch.long)
        )

    def forward(self, x):
        seq_len = x.size(1)
        pos = torch.arange(seq_len, dtype=torch.long).unsqueeze(0).expand_as(x).to(x.device)
        electron_mask = (x.unsqueeze(-1) == self.electron_token_indices).any(dim=-1)
        electron_mask = electron_mask.unsqueeze(-1).float()
        return self.norm(self.tok_embed(x) + self.pos_embed(pos) + electron_mask * self.electron_bias)


class ScaledDotProductAttention(nn.Module):
    def __init__(self, d_k):
        super().__init__(); self.d_k = d_k

    def forward(self, Q, K, V, attn_mask):
        scores = torch.matmul(Q, K.transpose(-1, -2)) / math.sqrt(self.d_k)
        scores.masked_fill_(attn_mask, -1e9)
        return torch.matmul(F.softmax(scores, dim=-1), V)


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, d_k, d_v, n_heads):
        super().__init__()
        self.d_k = d_k; self.d_v = d_v; self.n_heads = n_heads
        self.W_Q = nn.Linear(d_model, d_k * n_heads)
        self.W_K = nn.Linear(d_model, d_k * n_heads)
        self.W_V = nn.Linear(d_model, d_v * n_heads)
        self.linear = nn.Linear(n_heads * d_v, d_model)
        self.layernorm = nn.LayerNorm(d_model)

    def forward(self, Q, K, V, attn_mask):
        residual, batch_size = Q, Q.size(0)
        q_s = self.W_Q(Q).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        k_s = self.W_K(K).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        v_s = self.W_V(V).view(batch_size, -1, self.n_heads, self.d_v).transpose(1, 2)
        attn_mask = attn_mask.unsqueeze(1).repeat(1, self.n_heads, 1, 1)
        context = ScaledDotProductAttention(self.d_k)(q_s, k_s, v_s, attn_mask)
        context = context.transpose(1, 2).contiguous().view(batch_size, -1, self.n_heads * self.d_v)
        return self.layernorm(self.linear(context) + residual)


class PoswiseFeedForwardNet(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(d_model, d_ff, bias=False), nn.ReLU(), nn.Linear(d_ff, d_model, bias=False)
        )
        self.layernorm = nn.LayerNorm(d_model)

    def forward(self, inputs):
        return self.layernorm(self.fc(inputs) + inputs)


class EncoderLayer(nn.Module):
    def __init__(self, d_model, d_k, d_v, n_heads, d_ff):
        super().__init__()
        self.enc_self_attn = MultiHeadAttention(d_model, d_k, d_v, n_heads)
        self.pos_ffn = PoswiseFeedForwardNet(d_model, d_ff)

    def forward(self, enc_inputs, enc_self_attn_mask):
        return self.pos_ffn(self.enc_self_attn(enc_inputs, enc_inputs, enc_inputs, enc_self_attn_mask))


class BERT_atom_embedding_generator(nn.Module):
    def __init__(self, d_model, n_layers, vocab_size, maxlen, d_k, d_v, n_heads, d_ff,
                 global_label_dim, atom_label_dim,
                 bias_init=0.1, bias_learnable=True, electron_token_set=None):
        super().__init__()
        self.embedding = Embedding(
            vocab_size, d_model, maxlen,
            bias_init=bias_init,
            bias_learnable=bias_learnable,
            electron_token_set=electron_token_set,
        )
        self.layers = nn.ModuleList([
            EncoderLayer(d_model, d_k, d_v, n_heads, d_ff) for _ in range(n_layers)
        ])

    def forward(self, input_ids):
        output = self.embedding(input_ids)
        attn_mask = get_attn_pad_mask(input_ids)
        for layer in self.layers:
            output = layer(output, attn_mask)
        return output[:, 0, :]


_NUM_ATOM_TYPE = 119; _NUM_CHIRALITY_TAG = 3; _NUM_BOND_TYPE = 5; _NUM_BOND_DIRECTION = 3


def _gcn_norm(edge_index, num_nodes=None):
    from torch_geometric.utils.num_nodes import maybe_num_nodes
    num_nodes = maybe_num_nodes(edge_index, num_nodes)
    edge_weight = torch.ones(edge_index.size(1), device=edge_index.device)
    row, col = edge_index[0], edge_index[1]
    deg = scatter_add(edge_weight, col, dim=0, dim_size=num_nodes)
    deg_inv_sqrt = deg.pow_(-0.5)
    deg_inv_sqrt.masked_fill_(deg_inv_sqrt == float('inf'), 0)
    return edge_index, deg_inv_sqrt[row] * edge_weight * deg_inv_sqrt[col]


from torch_geometric.nn import MessagePassing
import torch_sparse


class GCNConv(MessagePassing):
    def __init__(self, emb_dim, aggr='add'):
        super().__init__()
        self.emb_dim = emb_dim; self.aggr = aggr
        self.weight = nn.Parameter(torch.Tensor(emb_dim, emb_dim))
        self.bias   = nn.Parameter(torch.Tensor(emb_dim))
        stdv = math.sqrt(6.0 / (emb_dim + emb_dim))
        self.weight.data.uniform_(-stdv, stdv); self.bias.data.fill_(0)
        self.edge_embedding1 = nn.Embedding(_NUM_BOND_TYPE, 1)
        self.edge_embedding2 = nn.Embedding(_NUM_BOND_DIRECTION, 1)
        nn.init.xavier_uniform_(self.edge_embedding1.weight.data)
        nn.init.xavier_uniform_(self.edge_embedding2.weight.data)

    def forward(self, x, edge_index, edge_attr):
        edge_index = add_self_loops(edge_index, num_nodes=x.size(0))[0]
        self_loop_attr = torch.zeros(x.size(0), 2, device=edge_attr.device, dtype=edge_attr.dtype)
        self_loop_attr[:, 0] = 4
        edge_attr = torch.cat((edge_attr, self_loop_attr), dim=0)
        edge_embeddings = self.edge_embedding1(edge_attr[:, 0]) + self.edge_embedding2(edge_attr[:, 1])
        edge_index, _ = _gcn_norm(edge_index)
        x = x @ self.weight
        return self.propagate(edge_index, x=x, edge_attr=edge_embeddings, size=None) + self.bias

    def message(self, x_j, edge_attr):
        return x_j if edge_attr is None else edge_attr + x_j

    def message_and_aggregate(self, adj_t, x):
        return torch_sparse.matmul(adj_t, x, reduce=self.aggr)


class GCN(nn.Module):
    def __init__(self, num_layer=5, emb_dim=300, feat_dim=300, drop_ratio=0.1, pool='mean'):
        super().__init__()
        self.num_layer = num_layer; self.drop_ratio = drop_ratio
        self.x_embedding1 = nn.Embedding(_NUM_ATOM_TYPE, emb_dim)
        self.x_embedding2 = nn.Embedding(_NUM_CHIRALITY_TAG, emb_dim)
        nn.init.xavier_uniform_(self.x_embedding1.weight.data)
        nn.init.xavier_uniform_(self.x_embedding2.weight.data)
        self.gnns = nn.ModuleList([GCNConv(emb_dim) for _ in range(num_layer)])
        self.batch_norms = nn.ModuleList([nn.BatchNorm1d(emb_dim) for _ in range(num_layer)])
        self.pool = global_mean_pool; self.feat_lin = nn.Linear(emb_dim, feat_dim)

    def forward(self, x, edge_index, edge_attr, batch):
        h = self.x_embedding1(x[:, 0]) + self.x_embedding2(x[:, 1])
        for i in range(self.num_layer):
            h = self.gnns[i](h, edge_index, edge_attr)
            h = self.batch_norms[i](h)
            h = F.dropout(h if i == self.num_layer - 1 else F.relu(h), self.drop_ratio, training=self.training)
        return self.feat_lin(self.pool(h, batch))


class CombinedModel(nn.Module):
    def __init__(self, one_model, two_model):
        super().__init__()
        self.one_model = one_model; self.two_model = two_model

    def forward(self, token_idx, x, edge_index, edge_attr, batch):
        return torch.cat((self.one_model(token_idx), self.two_model(x, edge_index, edge_attr, batch)), dim=1)


class FullModel(nn.Module):
    def __init__(self, combined_model, predictor):
        super().__init__()
        self.combined_model = combined_model; self.predictor = predictor

    def forward(self, token_idx, x, edge_index, edge_attr, batch):
        return self.predictor(self.combined_model(token_idx, x, edge_index, edge_attr, batch))


def build_esol_predictor(dim, dropout=0.1):
    # matches finetune_regression_ESOL.py exactly:
    # Linear -> Dropout -> ReLU -> Linear (no BatchNorm)
    return nn.Sequential(
        nn.Linear(dim, dim * 2), nn.Dropout(p=dropout), nn.ReLU(), nn.Linear(dim * 2, 1)
    )


# dataset
# dataset is loaded once and shared across all five variants.

_ATOM_LIST = list(range(1, 119))
_CHIRALITY_LIST = [
    Chem.rdchem.ChiralType.CHI_UNSPECIFIED,
    Chem.rdchem.ChiralType.CHI_TETRAHEDRAL_CW,
    Chem.rdchem.ChiralType.CHI_TETRAHEDRAL_CCW,
    Chem.rdchem.ChiralType.CHI_OTHER,
]
_BOND_LIST    = [BT.SINGLE, BT.DOUBLE, BT.TRIPLE, BT.AROMATIC]
_BONDDIR_LIST = [
    Chem.rdchem.BondDir.NONE,
    Chem.rdchem.BondDir.ENDUPRIGHT,
    Chem.rdchem.BondDir.ENDDOWNRIGHT,
]


@contextlib.contextmanager
def suppress_output():
    with open(os.devnull, 'w') as devnull:
        old_stdout, old_stderr = sys.stdout, sys.stderr
        sys.stdout = devnull; sys.stderr = devnull
        try:
            yield
        finally:
            sys.stdout = old_stdout; sys.stderr = old_stderr


class MolGraphDataset(Dataset):
    def __init__(self, csv_path, smiles_col, target_col):
        df = pd.read_csv(csv_path)
        self.smiles_raw  = df[smiles_col].tolist()
        self.targets_raw = df[target_col].tolist()
        self.valid_indices = [i for i in range(len(self.smiles_raw)) if self._is_valid(i)]
        self.smiles  = [self.smiles_raw[i]  for i in self.valid_indices]
        self.targets = [self.targets_raw[i] for i in self.valid_indices]
        print(f'  loaded {len(self.smiles)} / {len(self.smiles_raw)} valid molecules')

    def _is_valid(self, index):
        try:
            with suppress_output():
                encode_smiles(self.smiles_raw[index])
            return Chem.MolFromSmiles(self.smiles_raw[index]) is not None
        except:
            return False

    def __len__(self):
        return len(self.smiles)

    def __getitem__(self, index):
        smiles = self.smiles[index]
        target = float(self.targets[index])
        with suppress_output():
            token_seq = encode_smiles(smiles)
        tokens_idx, atom_mask = construct_token_sequence(token_seq, max_len=500)
        mol = Chem.AddHs(Chem.MolFromSmiles(smiles))
        type_idx, chirality_idx = [], []
        for atom in mol.GetAtoms():
            try:    type_idx.append(_ATOM_LIST.index(atom.GetAtomicNum()))
            except: type_idx.append(0)
            try:    chirality_idx.append(_CHIRALITY_LIST.index(atom.GetChiralTag()))
            except: chirality_idx.append(0)
        x = torch.cat([
            torch.tensor(type_idx,      dtype=torch.long).view(-1, 1),
            torch.tensor(chirality_idx, dtype=torch.long).view(-1, 1),
        ], dim=-1)
        row, col, edge_feat = [], [], []
        for bond in mol.GetBonds():
            s, e = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
            row += [s, e]; col += [e, s]
            try:    bt = _BOND_LIST.index(bond.GetBondType())
            except: bt = 0
            try:    bd = _BONDDIR_LIST.index(bond.GetBondDir())
            except: bd = 0
            edge_feat += [[bt, bd], [bt, bd]]
        if not row:
            row, col = [0], [0]; edge_feat = [[4, 0]]
        return (np.array(tokens_idx), np.array(atom_mask), np.array([target]),
                x, torch.tensor([row, col], dtype=torch.long),
                torch.tensor(edge_feat, dtype=torch.long))


def molgraph_collate_fn(data):
    batch_size = len(data)
    tokens_idx_batch = torch.zeros(batch_size, 501, dtype=torch.long)
    atom_mask_batch  = torch.zeros(batch_size, 501, dtype=torch.long)
    target_batch     = torch.zeros(batch_size, 1)
    x_list, edge_index_list, edge_attr_list, batch_list = [], [], [], []
    num_nodes = 0
    for i, (tokens_idx, atom_mask, target, x, edge_index, edge_attr) in enumerate(data):
        tokens_idx_batch[i] = torch.tensor(tokens_idx, dtype=torch.long)
        atom_mask_batch[i]  = torch.tensor(atom_mask,  dtype=torch.long)
        target_batch[i]     = torch.tensor(target)
        x_list.append(x); edge_index_list.append(edge_index + num_nodes)
        edge_attr_list.append(edge_attr)
        batch_list.append(torch.full((x.size(0),), i, dtype=torch.long))
        num_nodes += x.size(0)
    return (tokens_idx_batch, atom_mask_batch, target_batch,
            torch.cat(x_list, 0), torch.cat(edge_index_list, 1),
            torch.cat(edge_attr_list, 0), torch.cat(batch_list, 0))


def compute_splits(length, batch_size):
    # from finetune_regression_ESOL.py lines 93-102
    train = int(length * 0.8)
    test_guodu = length - train
    if test_guodu % batch_size != 0:
        test_guodu = (test_guodu // batch_size) * batch_size
        train = length - test_guodu
        if train % batch_size == 1:
            train += 2; test_guodu -= 2
    val = int(test_guodu / 2)
    return train, val, test_guodu - val


def run_eval(model, loader, device):
    # exact rmse as in finetune_regression_ESOL.py's run_eval function.
    model.eval()
    total_loss = 0.0; total_samples = 0
    with torch.no_grad():
        for token_idx, atom_mask, target, x, edge_index, edge_attr, batch in loader:
            token_idx  = token_idx.to(device); x = x.to(device)
            edge_index = edge_index.to(device); edge_attr = edge_attr.to(device)
            batch = batch.to(device); target = target.to(device)
            out = model(token_idx, x, edge_index, edge_attr, batch)
            if torch.isnan(out).any(): continue
            total_loss    += F.mse_loss(out, target, reduction='sum').item()
            total_samples += target.numel()
    return float(np.sqrt(total_loss / max(total_samples, 1)))


# training function

def run_esol_variant(variant, full_dataset, train_indices, val_indices, test_indices, device):
    # trains one ablation variant on the pre-loaded esol dataset.
    # variant: a dict from ABLATION_VARIANTS.
    # returns a result dict for the summary table.

    vname = variant['name']
    print(f'\n{"=" * 64}')
    print(f'variant {variant["id"]}: {vname}')
    print(f'  bias_init    : {variant["bias_init"]}')
    print(f'  bias_learnable: {variant["bias_learnable"]}')
    print(f'  token_set    : {"electron (correct)" if variant["token_set"] == ELECTRON_RELEVANT_TOKEN_SET else "inert (wrong)"}')
    print(f'  gcn_layers   : {variant["gcn_layers"]}')
    print(f'  predictor_drop: {variant["predictor_drop"]}')
    print(f'{"=" * 64}')

    torch.manual_seed(ESOL_SEED)
    np.random.seed(ESOL_SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(ESOL_SEED)

    train_dataset = Subset(full_dataset, train_indices)
    val_dataset   = Subset(full_dataset, val_indices)
    test_dataset  = Subset(full_dataset, test_indices)

    train_loader = DataLoader(train_dataset, batch_size=ESOL_BS, shuffle=True,
                              collate_fn=molgraph_collate_fn)
    val_loader   = DataLoader(val_dataset,   batch_size=ESOL_BS,
                              collate_fn=molgraph_collate_fn)
    test_loader  = DataLoader(test_dataset,  batch_size=ESOL_BS,
                              collate_fn=molgraph_collate_fn)

    # build model with variant-specific embedding parameters
    one_model = BERT_atom_embedding_generator(
        d_model=D_MODEL, n_layers=N_LAYERS, vocab_size=VOCAB_SIZE,
        maxlen=MAXLEN, d_k=D_K, d_v=D_V, n_heads=N_HEADS, d_ff=D_FF,
        global_label_dim=1, atom_label_dim=15,
        bias_init=variant['bias_init'],
        bias_learnable=variant['bias_learnable'],
        electron_token_set=variant['token_set'],
    )
    two_model = GCN(
        num_layer=variant['gcn_layers'], emb_dim=GCN_EMB_DIM,
        feat_dim=GCN_FEAT_DIM, drop_ratio=variant['gcn_drop'], pool='mean',
    )
    model = FullModel(
        CombinedModel(one_model, two_model),
        build_esol_predictor(COMBINED_DIM, dropout=variant['predictor_drop'])
    ).to(device)

    criterion = nn.MSELoss()

    # no weight_decay, matching finetune_regression_ESOL.py line 131
    optimizer = optim.Adam(model.parameters(), lr=ESOL_LR, betas=(0.9, 0.98))
    lr_scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, 'min', factor=ESOL_LR_FACTOR,
        patience=ESOL_LR_PATIENCE, min_lr=ESOL_LR_MIN
    )

    best_metric     = 1e9
    best_epoch      = 0
    best_model      = None
    early_stop      = 0
    test_rmse_list  = []
    t0              = time.time()

    for epoch in range(ESOL_EPOCHS):
        model.train()
        train_loss = 0.0
        t1 = time.time()

        for token_idx, atom_mask, target, x, edge_index, edge_attr, batch in train_loader:
            token_idx  = token_idx.to(device); atom_mask = atom_mask.to(device)
            target     = target.to(device); x = x.to(device)
            edge_index = edge_index.to(device); edge_attr = edge_attr.to(device)
            batch      = batch.to(device)
            optimizer.zero_grad()
            out = model(token_idx, x, edge_index, edge_attr, batch)
            if torch.isnan(out).any(): continue
            loss_batch = criterion(out, target.float())
            train_loss += loss_batch.item() / (len(target) * ESOL_BS)
            loss_batch.backward()
            optimizer.step()

        val_rmse  = run_eval(model, val_loader,  device)
        test_rmse = run_eval(model, test_loader, device)
        test_rmse_list.append(test_rmse)
        lr_scheduler.step(val_rmse)

        # read electron_bias. for ablation a: always 0. for c: always 0.1.
        # for ablation b: moving, but on the wrong tokens.
        bias_val = model.combined_model.one_model.embedding.electron_bias.item()

        print(
            f'  epoch {epoch+1:03d}/{ESOL_EPOCHS} | '
            f'loss: {train_loss*1e4:.2f} | '
            f'val_rmse: {val_rmse:.4f} | '
            f'test_rmse: {test_rmse:.4f} | '
            f'lr: {optimizer.param_groups[0]["lr"]:.2e} | '
            f'e_bias: {bias_val:.5f} | '
            f't: {time.time()-t1:.1f}s'
        )

        if val_rmse < best_metric:
            best_metric = val_rmse; best_model = copy.deepcopy(model)
            best_epoch = epoch + 1; early_stop = 0
        else:
            early_stop += 1

        if early_stop >= ESOL_EARLY_STOP:
            print(f'  early stopping: no improvement for {ESOL_EARLY_STOP} epochs.')
            break

    elapsed = time.time() - t0
    final_test_rmse = run_eval(best_model, test_loader, device)
    best_test_rmse  = min(test_rmse_list)
    learned_bias    = best_model.combined_model.one_model.embedding.electron_bias.item()

    print(f'  best epoch    : {best_epoch}')
    print(f'  val rmse      : {best_metric:.5f}')
    print(f'  test rmse     : {final_test_rmse:.5f}')
    print(f'  learned e_bias: {learned_bias:.6f}')
    print(f'  time          : {elapsed/60:.1f} min')

    return {
        'variant_id':       variant['id'],
        'variant_name':     variant['short'],
        'bias_init':        variant['bias_init'],
        'bias_learnable':   variant['bias_learnable'],
        'token_set':        'electron' if variant['token_set'] == ELECTRON_RELEVANT_TOKEN_SET else 'inert',
        'gcn_layers':       variant['gcn_layers'],
        'predictor_drop':   variant['predictor_drop'],
        'best_epoch':       best_epoch,
        'best_val_rmse':    best_metric,
        'final_test_rmse':  final_test_rmse,
        'best_test_rmse':   best_test_rmse,
        'learned_bias':     learned_bias,
        'training_min':     elapsed / 60,
    }



def main():
    torch.manual_seed(ESOL_SEED)
    np.random.seed(ESOL_SEED)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    print(f'device: {device}')
    print(f'running {len(ABLATION_VARIANTS)} ablation variants on esol')
    print()

    # load the dataset once. all five variants use the same encoded molecules.
    # this saves ~16s × 4 = ~64 seconds vs loading once per variant.
    print('loading and encoding esol dataset (shared across all variants)...')
    t0 = time.time()
    full_dataset = MolGraphDataset(
        ESOL_CSV,
        smiles_col='smiles',
        target_col='measured log solubility in mols per litre',
    )
    print(f'  ready in {time.time()-t0:.1f}s')

    length = len(full_dataset)
    indices = list(range(length))
    train_n, val_n, test_n = compute_splits(length, ESOL_BS)
    print(f'  split: train={train_n}, val={val_n}, test={test_n}')

    # use a fixed seed for the split so all variants see the same
    # train/val/test partition. this isolates the ablation effect.
    generator = torch.Generator().manual_seed(ESOL_SEED)
    train_indices, val_indices, test_indices = random_split(
        indices, [train_n, val_n, test_n], generator=generator
    )

    all_results = []
    for variant in ABLATION_VARIANTS:
        result = run_esol_variant(
            variant, full_dataset,
            train_indices, val_indices, test_indices,
            device
        )
        all_results.append(result)

    # print comparison table
    print()
    print('=' * 80)
    print('esol ablation study — final comparison')
    print('all variants use identical training config from finetune_regression_ESOL.py')
    print('=' * 80)
    hdr = f'  {"variant":<18} {"val rmse":>10} {"test rmse":>10} {"e_bias":>10} {"best ep":>8}'
    print(hdr)
    print('  ' + '-' * (len(hdr) - 2))
    for r in all_results:
        print(
            f'  {r["variant_name"]:<18} '
            f'{r["best_val_rmse"]:>10.5f} '
            f'{r["final_test_rmse"]:>10.5f} '
            f'{r["learned_bias"]:>10.6f} '
            f'{r["best_epoch"]:>8}'
        )
    print('=' * 80)
    print()

    # ablation interpretation guidance for the paper
    baseline_rmse = next(r['final_test_rmse'] for r in all_results if r['variant_name'] == 'baseline')
    null_rmse     = next(r['final_test_rmse'] for r in all_results if r['variant_name'] == 'no_bias')
    wrong_rmse    = next(r['final_test_rmse'] for r in all_results if r['variant_name'] == 'wrong_chem')
    static_rmse   = next(r['final_test_rmse'] for r in all_results if r['variant_name'] == 'static_bias')

    print('ablation interpretation:')
    print(f'  baseline vs null (ablation a): delta = {null_rmse - baseline_rmse:+.5f}')
    print(f'    → positive delta = bias helps. negative = bias hurts.')
    print(f'  baseline vs wrong chem (ablation b): delta = {wrong_rmse - baseline_rmse:+.5f}')
    print(f'    → positive delta = correct token set matters.')
    print(f'  baseline vs static (ablation c): delta = {static_rmse - baseline_rmse:+.5f}')
    print(f'    → positive delta = the gradient-updated bias was better than frozen.')

    df = pd.DataFrame(all_results)
    df.to_csv('esol_ablation_results.csv', index=False)
    print()
    print('results saved to esol_ablation_results.csv')


if __name__ == '__main__':
    main()


installing dependencies...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 23.0 MB/s eta 0:00:00
  using pyg wheel index: https://data.pyg.org/whl/torch-2.10.0+cu128.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 83.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 49.9 MB/s eta 0:00:00
  installing rdkit...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.0/37.0 MB 52.0 MB/s eta 0:00:00
  nltk: already installed, skipping.
  six: already installed, skipping.
all dependencies installed.
device: cuda
running 5 ablation variants on esol

loading and encoding esol dataset (shared across all variants)...
  loaded 1035 / 1128 valid molecules
  ready in 15.8s
  split: train=843, val=96, test=96

variant 0: baseline (learnable bias, correct chemistry)
  bias_init    : 0.1
  bias_learnable: True
  token_set    : electron (correct)
  gcn_layers   : 5
  predictor_drop: 